# Parametric primitives (C++)

The C++ counterpart of [`../python/40_primitives.ipynb`](../python/40_primitives.ipynb).
`operations/primitives.hpp` builds a shape from parameters, beside `grid`: `rectangle`, `box`,
`disk`, `uv_sphere`, `icosa_sphere`, `cylinder`, the planar domains (`reference_triangle`, `ngon`, `annulus`, `lshape`, `plate_with_hole`, `honeycomb`) and `reference_cells`. They keep `grid`'s contract
(coordinates computed from indices, a `max_cells` budget, an empty mesh for a zero count), orient
surfaces outward and volumes positively, return the boundary as named `Side` regions, and with
`Order == 2` place the mid-edge nodes on the exact geometry. See [parametric primitives](../../doc/primitives.md).

In [ ]:
#define MESHIOPLUSPLUS_IMPLEMENTATION
#include "../../src/single_include/meshioplusplus/meshioplusplus.hpp"
#include "mio_notebook.hpp"
#include <cmath>
#include <iostream>
#include <map>
#include <set>
using namespace meshioplusplus;
using namespace mio_nb;

// What a generated mesh holds: its points, each cell block and each named side region.
void summarize(const char* title, const Mesh& m) {
    std::cout << title << ": " << m.NumPoints() << " points";
    for (std::size_t b = 0; b < m.NumCellBlocks(); ++b)
        std::cout << ", " << m.Cells(b).NumCells() << " " << m.Cells(b).Type();
    for (std::size_t r = 0; r < m.NumRegions(); ++r)
        std::cout << (r == 0 ? "\n  regions:" : ",") << " " << m.Region(r).mName << "=" << m.Region(r).NumEntries();
    std::cout << "\n";
}

// The SVG writer draws linear cells, so a quadratic mesh is drawn tessellated: every curved cell is
// cut into many small straight ones that follow the exact geometry.
Mesh drawable(const Mesh& m, int levels = 3) {
    for (std::size_t b = 0; b < m.NumCellBlocks(); ++b) {
        const std::string type = m.Cells(b).Type();
        if (type == "triangle6" || type == "quad8" || type == "tetra10" || type == "hexahedron20" || type == "wedge15") {
            Mesh bare = m;  // tessellate drops side regions with a warning; the drawing does not need them
            while (bare.NumRegions() > 0) bare.RemoveRegion(0);
            TessellateOptions o;
            o.mLevels = levels;
            return tessellate(bare, o).mMesh;
        }
    }
    return m;
}

## Rectangles and their triangle layouts

`rectangle(dims, size, origin, cellType, split, order, maxCells)` lays out a 2-D grid at `z = origin[2]`.
Triangles split each quad `zigzag` (alternating diagonals, the default), `up`, `down`, or `center`
(four triangles around an added centre node).

In [ ]:
for (const char* split : {"zigzag", "up", "down", "center"}) {
    const Mesh m = rectangle({{4, 3}}, {{4.0, 3.0}}, {{0.0, 0.0, 0.0}}, "triangle", split);
    summarize(split, m);
}
const Mesh quads = rectangle({{8, 4}}, {{2.0, 1.0}});
summarize("quads", quads);
xcpp::display(mio_nb::render(rectangle({{4, 3}}, {{4.0, 3.0}}, {{0.0, 0.0, 0.0}}, "triangle", "center")));

zigzag: 20 points, 24 triangle
  regions: xmax=3, xmin=3, ymax=4, ymin=4
up: 20 points, 24 triangle
  regions: xmax=3, xmin=3, ymax=4, ymin=4
down: 20 points, 24 triangle
  regions: xmax=3, xmin=3, ymax=4, ymin=4
center: 32 points, 48 triangle
  regions: xmax=3, xmin=3, ymax=4, ymin=4
quads: 45 points, 32 quad
  regions: xmax=4, xmin=4, ymax=8, ymin=8


## Boxes and their six side regions

`box` builds hexahedra, **tetrahedra** (the six Kuhn tetrahedra of each hexahedron around its main
diagonal, which conform across cells) or **wedges**. The boundary comes back as the named `Side`
regions `xmin` ... `zmax`; each entry is a `(cell, local facet)` pair.

In [ ]:
for (const char* type : {"hexahedron", "tetra", "wedge"})
    summarize(type, box({{2, 2, 2}}, {{1.0, 1.0, 1.0}}, {{0.0, 0.0, 0.0}}, type));

const Mesh cube = box({{3, 3, 3}});
const Region& xmin = cube.Region(cube.FindRegion("xmin", RegionKind::Side));
const std::int64_t* entries = xmin.mEntries.As<std::int64_t>();
std::cout << "xmin entries (cell, facet):";
for (std::size_t i = 0; i < 3; ++i) std::cout << " (" << entries[2 * i] << ", " << entries[2 * i + 1] << ")";
std::cout << " ...\n";
xcpp::display(mio_nb::render(box({{2, 2, 2}}, {{1.0, 1.0, 1.0}}, {{0.0, 0.0, 0.0}}, "tetra")));

hexahedron: 27 points, 8 hexahedron
  regions: xmax=4, xmin=4, ymax=4, ymin=4, zmax=4, zmin=4
tetra: 27 points, 48 tetra
  regions: xmax=8, xmin=8, ymax=8, ymin=8, zmax=8, zmin=8
wedge: 27 points, 16 wedge
  regions: xmax=4, xmin=4, ymax=4, ymin=4, zmax=8, zmin=8
xmin entries (cell, facet): (0, 0) (3, 0) (6, 0) ...


## Disks, spheres and cylinders

`disk` pushes an `n x n` grid onto the circle with the squircle map, or builds `n` rings of triangles.
`uv_sphere` and `icosa_sphere` are oriented outward with welded poles and seams, and `cylinder` extrudes
a disk into hexahedra or wedges and names its `side`, `top` and `bottom` regions.

In [ ]:
summarize("disk (quads)", disk(1.0, 6));
summarize("disk (triangles)", disk(1.0, 4, {{0.0, 0.0, 0.0}}, "triangle"));
summarize("uv_sphere", uv_sphere(1.0, 20, 10));
summarize("uv_sphere (quads)", uv_sphere(1.0, 20, 10, {{0.0, 0.0, 0.0}}, "quad"));
summarize("icosa_sphere", icosa_sphere(1.0, 2));
summarize("cylinder", cylinder(0.6, 2.0, 4, 6));
xcpp::display(mio_nb::render(disk(1.0, 6)));
xcpp::display(mio_nb::render(icosa_sphere(1.0, 2)));
xcpp::display(mio_nb::render(cylinder(0.6, 2.0, 4, 6, {{0.0, 0.0, 0.0}}, "wedge")));

disk (quads): 49 points, 36 quad
  regions: rim=24
disk (triangles): 61 points, 96 triangle
  regions: rim=24
uv_sphere: 182 points, 360 triangle
uv_sphere (quads): 182 points, 160 quad, 40 triangle
icosa_sphere: 162 points, 320 triangle
cylinder: 175 points, 96 hexahedron
  regions: bottom=16, side=96, top=16


## Planar domains

Six more generators build the 2-D domains that refinement, error estimation and stress studies start from, at `z = origin[2]`:
`reference_triangle` (named so because `meshioplusplus::triangle` would be the Triangle format's namespace-neighbour), `ngon`, `annulus`
(or a sector of one whose end rays stay separate node chains, the seam pair `periodic` links), `lshape`, `plate_with_hole` and
`honeycomb` (hexagons as one ragged `polygon` block). Each boundary is a named side region.

In [ ]:
summarize("reference_triangle(4)", reference_triangle(4));
summarize("ngon(7, rings=2)", ngon(7, 1.0, 2));
summarize("annulus (full ring)", annulus(0.5, 1.0, 24, 3));
summarize("annulus (sector, angle = pi)", annulus(0.5, 1.0, 12, 3, M_PI));
summarize("lshape(4)", lshape(4));
summarize("plate_with_hole", plate_with_hole(0.25, 1.0, 6, 4));
summarize("honeycomb(5 x 4)", honeycomb({{5, 4}}, 0.5));
xcpp::display(mio_nb::render(lshape(4, 2.0, {{0.0, 0.0, 0.0}}, "triangle")));
xcpp::display(mio_nb::render(plate_with_hole(0.25, 1.0, 6, 4)));
xcpp::display(mio_nb::render(annulus(0.5, 1.0, 12, 3, M_PI)));

reference_triangle(4): 15 points, 16 triangle
  regions: hypotenuse=4, xmin=4, ymin=4
ngon(7, rings=2): 22 points, 28 triangle
  regions: rim=14
annulus (full ring): 96 points, 72 quad
  regions: inner=24, outer=24
annulus (sector, angle = pi): 52 points, 36 quad
  regions: end=3, inner=12, outer=12, start=3
lshape(4): 65 points, 48 quad
  regions: notch_x=4, notch_y=4, xmax=4, xmin=8, ymax=8, ymin=4
  regions: hole=24, xmax=6, xmin=6, ymax=6, ymin=6
honeycomb(5 x 4): 58 points, 20 polygon
  regions: boundary=34


### What each shape promises

Counter-clockwise cells (the smallest signed area is positive), exact areas where the geometry is polygonal, welded seams on a full ring
and a seam pair on the sector.

In [ ]:
auto area = [](const Mesh& m) {
    double total = 0.0, smallest = 1e300;
    const double* p = m.Points().As<double>();
    for (std::size_t b = 0; b < m.NumCellBlocks(); ++b) {
        const auto view = m.Cells(b);
        for (std::size_t c = 0; c < view.NumCells(); ++c) {
            std::vector<std::int64_t> ids;
            if (view.IsRagged()) ids.assign(view.Row(c), view.Row(c) + view.RowSize(c));
            else {
                const std::int64_t* row = view.Conn().As<std::int64_t>() + c * view.NodesPerCell();
                ids.assign(row, row + (view.Type().rfind("triangle", 0) == 0 ? 3 : 4));
            }
            double a = 0.0;
            for (std::size_t k = 0; k < ids.size(); ++k) {
                const double* u = p + 3 * ids[k];
                const double* v = p + 3 * ids[(k + 1) % ids.size()];
                a += u[0] * v[1] - v[0] * u[1];
            }
            total += a / 2;
            smallest = std::min(smallest, a / 2);
        }
    }
    return std::make_pair(total, smallest);
};
std::cout << std::fixed << std::setprecision(4);
for (const auto& [title, m] : std::vector<std::pair<const char*, Mesh>>{
         {"lshape", lshape(4)}, {"plate_with_hole", plate_with_hole(0.25, 1.0, 6, 4)},
         {"annulus", annulus(0.5, 1.0, 24, 3)}, {"honeycomb", honeycomb({{5, 4}}, 0.5)}})
    std::cout << title << ": area " << area(m).first << ", smallest cell " << area(m).second << "\n";
std::cout << "full ring welds its seam: " << annulus(0.5, 1.0, 24, 3).NumPoints() << " points (24 x 4); a sector keeps both end rays: "
          << annulus(0.5, 1.0, 12, 3, M_PI).NumPoints() << " (13 x 4)\n";

plate_with_hole: 120 points, 96 quad
lshape: area 3.0000, smallest cell 0.0625
plate_with_hole: area 0.8059, smallest cell 0.0050
annulus: area 2.3294, smallest cell 0.0252
honeycomb: area 12.9904, smallest cell 0.6495
full ring welds its seam: 96 points (24 x 4); a sector keeps both end rays: 52 (13 x 4)


## `Order == 2`: mid-edge nodes on the exact geometry

With `Order == 2` a generator returns serendipity cells (`triangle6`, `quad8`, `tetra10`, `hexahedron20`,
`wedge15`). Every mid-edge node is evaluated from the shape's own map, never as a chord midpoint, so a
coarse quadratic sphere is genuinely round. Here every node of a 20-triangle icosphere is on the sphere
of radius 2, and the cells are drawn tessellated so the curvature is visible.

In [ ]:
auto radii = [](const Mesh& m) {
    double lo = 1e300, hi = -1e300;
    const double* p = m.Points().As<double>();
    for (std::size_t i = 0; i < m.NumPoints(); ++i) {
        const double r = std::sqrt(p[3 * i] * p[3 * i] + p[3 * i + 1] * p[3 * i + 1] + p[3 * i + 2] * p[3 * i + 2]);
        lo = std::min(lo, r);
        hi = std::max(hi, r);
    }
    return std::make_pair(lo, hi);
};
const Mesh coarse = icosa_sphere(2.0, 0);
const Mesh curved = icosa_sphere(2.0, 0, {{0.0, 0.0, 0.0}}, 2);
std::cout << std::fixed << std::setprecision(6);
std::cout << "order 1: " << coarse.NumPoints() << " points, radii " << radii(coarse).first << " .. " << radii(coarse).second << "\n";
std::cout << "order 2: " << curved.NumPoints() << " points, radii " << radii(curved).first << " .. " << radii(curved).second << "\n";
xcpp::display(mio_nb::render(coarse));
xcpp::display(mio_nb::render(drawable(curved)));
xcpp::display(mio_nb::render(drawable(cylinder(1.0, 1.0, 2, 2, {{0.0, 0.0, 0.0}}, "hexahedron", 2))));

order 1: 12 points, radii 2.000000 .. 2.000000
order 2: 42 points, radii 2.000000 .. 2.000000


## Reference cells

`reference_cells(types)` returns one cell of each requested type in its reference coordinates, laid out side
by side without touching: the cheapest way to show a format's cell coverage. With `pairs` it returns two
cells that share a facet.

In [ ]:
const Mesh ref = reference_cells({"triangle", "quad", "tetra", "hexahedron", "wedge", "pyramid",
                                  "triangle6", "quad8", "tetra10", "hexahedron20", "wedge15", "pyramid13"});
for (std::size_t b = 0; b < ref.NumCellBlocks(); ++b) std::cout << ref.Cells(b).Type() << " ";
std::cout << "\n";
std::cout << "polyhedron block: " << reference_cells({"polyhedron"}).Cells(0).Type() << "\n";
const Mesh pairs = reference_cells({"tetra", "hexahedron"}, true);
for (std::size_t b = 0; b < pairs.NumCellBlocks(); ++b)
    std::cout << "pair of " << pairs.Cells(b).Type() << ": " << pairs.Cells(b).NumCells() << " cells\n";
xcpp::display(mio_nb::render(drawable(reference_cells({"tetra", "hexahedron", "wedge", "pyramid", "tetra10", "hexahedron20"}))));

triangle quad tetra hexahedron wedge pyramid triangle6 quad8 tetra10 hexahedron20 wedge15 pyramid13 
polyhedron block: polyhedron8
pair of tetra: 2 cells
pair of hexahedron: 2 cells


## Cross-checks and the contract

The generators are cheap, so they make good oracles. A `box` is `grid` with the same lattice, a surface is
closed, and an oversized request is refused by name instead of running out of memory.

In [ ]:
const Mesh b = box({{4, 3, 2}}, {{4.0, 3.0, 2.0}});
const Mesh g = grid({{4, 3, 2}});
bool same = b.NumPoints() == g.NumPoints() && b.Cells(0).NumCells() == g.Cells(0).NumCells();
for (std::size_t i = 0; same && i < b.NumPoints() * 3; ++i) same = b.Points().As<double>()[i] == g.Points().As<double>()[i];
const std::size_t nconn = g.Cells(0).NumCells() * 8;
for (std::size_t i = 0; same && i < nconn; ++i) same = b.Cells(0).Conn().As<std::int64_t>()[i] == g.Cells(0).Conn().As<std::int64_t>()[i];
std::cout << "box == grid: " << (same ? "true" : "false") << "\n";

const Mesh sphere = icosa_sphere(1.0, 3);
const std::int64_t* tri = sphere.Cells(0).Conn().As<std::int64_t>();
const double* p = sphere.Points().As<double>();
std::map<std::pair<std::int64_t, std::int64_t>, int> edges;
double volume = 0.0;
for (std::size_t f = 0; f < sphere.Cells(0).NumCells(); ++f) {
    for (int e = 0; e < 3; ++e) {
        std::int64_t u = tri[3 * f + e], v = tri[3 * f + (e + 1) % 3];
        ++edges[{std::min(u, v), std::max(u, v)}];
    }
    const double* a = p + 3 * tri[3 * f];
    const double* bb = p + 3 * tri[3 * f + 1];
    const double* c = p + 3 * tri[3 * f + 2];
    volume += (a[0] * (bb[1] * c[2] - bb[2] * c[1]) - a[1] * (bb[0] * c[2] - bb[2] * c[0]) + a[2] * (bb[0] * c[1] - bb[1] * c[0])) / 6.0;
}
bool closed = true;
for (const auto& kv : edges) closed = closed && kv.second == 2;
std::cout << "icosa_sphere closed: " << (closed ? "true" : "false") << ", enclosed volume " << volume
          << " (4/3 pi = " << 4.0 / 3.0 * M_PI << ")\n";

try {
    box({{1000, 1000, 1000}});
} catch (const std::invalid_argument& e) {
    std::cout << e.what() << "\n";
}

box == grid: true
icosa_sphere closed: true, enclosed volume 4.152741 (4/3 pi = 4.188790)
meshio++: box: the requested mesh has 1000000000 cells, above the limit of 20000000 (raise max_cells or coarsen the counts)


## Beyond C++

The same generators are available from Python, C (`mio_rectangle` ... `mio_honeycomb`, `mio_reference_cells`), Fortran, Julia,
R and WASM, as `meshioplusplus generate box block.vtu --dims 8,8,8 --cell-type tetra` on both command lines,
as MCP tools, and as a pipeline source (`{ "Op": "Generate", "Shape": "box" }`, which needs no `Input`).